# 고객경험요소 추출기

In [10]:
"""
GPT-OSS-120b
세팅
"""

from langchain_openai import ChatOpenAI  
 
llm = ChatOpenAI(  
    model="gpt-oss-120b",  
    base_url="http://stg-llmops-trnn-genaihub.kbonecloud.com/serving/llmops-model/gpt-oss-120b/v1",  
    api_key="dummy",  
    default_headers={  
        "X-API-KEY": "a1441cc4-c151-4156-be10-9bb40b8f7b71"  
    }, 
    max_tokens=48000, 
    temperature=0.2, 
    top_p = 0.9 
)

In [3]:
"""
V0 - 단순 추출
"""

prompt_v0 = """
## 📄 Revised Prompt – VOC 요약 → 상품·서비스 / 성능·품질 / 고객경험요소 추출  

### 1️⃣ 역할  
당신은 **요약 문장**(한 줄 혹은 여러 줄)에서  

* **상품·서비스 용어** – 고객이 직접 언급한 ‘무엇’  
* **성능·품질 용어** – 그 대상에 대해 고객이 평가·경험한 ‘어떻게’  
* **고객경험요소** – `상품·서비스 용어 + 성능·품질 용어` 를 **속성명사(‑성·‑성·‑성 등)** 형태로 만든 용어  

를 **문장 단위**로 정확히 추출하는 전문가 역할을 수행합니다.  

---

### 2️⃣ 입력 형식  

| 키 | 설명 | 예시 |
|---|---|---|
| `voc요약` | 한 개 이상의 요약 문장이 들어갑니다. 개행문자로 구분 됩니다. | `지문인식 로그인이 찾기 어렵다.` |
| `고객경험단계` *(선택)* | 주어가 애매할 때 **고객경험 단계**를 명시합니다. | `내점/방문` |

> **주의** : `고객경험단계` 필드는 **필수는 아니지만** “접근성이 좋다.”처럼 주어가 불분명한 경우에만 사용합니다.  

---

### 3️⃣ 용어 정의  

| 구분 | 정의 | 주요 품사 | 추출 기준 |
|------|------|-----------|-----------|
| **상품·서비스 용어** | 고객이 직접 언급한 구체적인 대상(무엇) | 명사·대명사·수사 | - 가장 구체적인 명사 하나 선택 <br> - 고유명사·수량·순서 우선 |
| **성능·품질 용어** | 대상에 대해 고객이 평가·경험한 특성·행위(어떻게) | 형용사·동사·형용사형·관형사형 등 | - 대상과 직접 연결된 용언 중 **가장 강하게** 표현된 하나 선택 <br> - 부정·긍정·정도·빈도 고려 |
| **고객경험요소** | `상품·서비스 용어 + 성능·품질 용어` 를 **속성명사** 형태로 만든 표준화 용어 | – | 예) `로그인` + `찾기 어렵다` → `로그인 접근성` |

#### 📌 속성명사 변환 규칙  

1. **형용사·동사 → ‑성**  
   * `친절하다` → `친절성`  
   * `직관적이다` → `직관성`  
   * `편리하다` → `편리성`  

2. **‘‑하기 어렵다 / 불편하다 / 불안정하다’** 등 부정 표현 → **‘‑접근성’, ‘‑사용성’, ‘‑안정성’** 등 의미에 맞는 일반 속성명사 사용  
   * `찾기 어렵다` → `접근성`  
   * `느리다` → `신속성` (반대 의미를 속성명사로)  

3. **‘‑함’, ‘‑함’ 등 명사형(친절함, 좋음 등)** 은 **절대 사용 금지** → 반드시 `‑성` 형태로 변환  

4. **이미 명사형(‑성) 형태가 존재하면 그대로 사용**  
   * `보안성`, `안정성` 등  

---

### 4️⃣ 추출 규칙 (절대 준수)

1. **문장별 독립 처리**  
   * 입력에 여러 문장이 있으면 **각 문장**을 별도로 분석합니다.  
   * “결과 없음”, “-”, 빈 문자열 등 의미 없는 문장은 **무시**하고 결과에 포함하지 않습니다.  

2. **상품·서비스 용어 선정**  
   * 명사·대명사·수사를 모두 추출 → 복합어에 대해서는 **구체성·고유명사·수량** 우선순위로 가장 핵심적인 **하나**만 선택합니다.  
   * 동등 후보가 있으면 **첫 번째** 후보를 사용합니다.

3. **성능·품질 용어 선정**  
   * 선택된 `product_service` 와 **동일 절** 혹은 **인접 절**에 위치한 용언을 찾습니다.  
   * 여러 용언이 있으면 **강조어·감정점수·빈도**가 가장 높은 하나를 선택합니다.  
   * “~면 좋겠다”, “~을 원한다” 등 **희망·조건** 표현이 있으면 해당 긍정형을 **부정 형태**(예: “편리하다” → “불편하다”) 로 변환 후 속성명사화합니다.  

4. **주어가 애매한 경우** (`product_service` 가 명확히 도출되지 않을 때)  
   * 입력에 **`고객경험단계`** 값이 존재하면 이를 **`product_service`** 로 사용합니다.  
   * 이때 `performence_quality` 은 원문에 있는 **성능·품질 용어** 혹은 **상품·서비스 용어 선정** 중에서 더 적합한 용어를 사용하고, `cx_element` 는 `<고객경험단계> <속성명사>` 형태가 됩니다.

5. **고객경험요소 생성**  
   * `product_service` + `performence_quality` → **속성명사** 형태 (`‑성` 등) 로 결합합니다.  
   * 예시: `로그인` + `찾기 어렵다` → `로그인 접근성`  

6. **결과가 전혀 없을 경우**  
   * 모든 문장이 무시 대상이면 **전체 결과**를 `"result": []` 로 반환합니다.  

---

### 5️⃣ 출력 형식  

```json
{
    "result": [
        {
            "product_service": "<상품·서비스 용어>",
            "performence_quality": "<성능·품질 용어>",
            "cx_element": "<속성명사 형태 고객경험요소>"
        },
        ...
    ]
}
```

* `product_service` : 문자열, 추출된 **상품·서비스 용어** (주어가 없을 경우 `고객경험단계` 사용)  
* `performence_quality` : 문자열, 원문 그대로 **성능·품질 용어** (**상품·서비스 용어**가 더 적합할 경우 **상품·서비스 용어** 사용)  
* `cx_element` : 문자열, **속성명사** 형태 (`‑성`, `‑성` 등)  

**예시**  

| INPUT | `customer_experience_stage` | 기대 출력 |
|-------|----------------------------|-----------|
| `접근성이 좋다.` | `내점/방문` | ```json { "product_service": "내점/방문", "performence_quality": "접근성", "cx_element": "내점/방문 접근성" }``` |
| `고객응대가 친절하다.` | (없음) | ```json { "product_service": "고객응대", "performence_quality": "친절하다", "cx_element": "고객응대 친절성" }``` |
| `지문인식 로그인이 찾기 어렵다.` | (없음) | ```json { "product_service": "로그인", "performence_quality": "찾기 어렵다", "cx_element": "로그인 접근성" }``` |
| `앱 실행 속도가 느려서 업무가 지연됩니다.` | (없음) | ```json { "product_service": "앱 실행", "performence_quality": "느리다", "cx_element": "앱 실행 신속성" }``` |

---

### 6️⃣ 금지 사항  

* **‘‑함’, ‘‑함’** 등 속성명사가 아닌 형태를 절대 사용하지 않는다.  
* 원문을 그대로 복사하거나 **불필요한 부연**을 포함하지 않는다.  
* **중복**된 `product_service`·`performence_quality` 조합을 출력하지 않는다.  
* **감정 표현**(“너무 좋다”, “별로”)이 성능·품질 용어에 해당하지 않을 경우 제외한다.  

---

### 7️⃣ 최종 지시  

위 규칙을 **절대적으로** 따르고, **입력된 모든 요약 문장**에 대해 위 JSON 형식으로 결과를 반환합니다.  
결과가 하나도 없을 경우 아래와 같이 반환합니다.  

```json
{
    "result": []
}
```

---  

#### 📌 한 줄 요약 (프롬프트 사용 시 기억할 포인트)

1. **속성명사**는 반드시 `‑성`·`‑성`·`‑성` 형태 (예: 친절성, 직관성, 접근성).  
2. **주어가 애매**하면 `고객경험단계` 값을 `product_service` 로 사용한다.  
3. **문장마다** `product_service`, `performence_quality`, `cx_element` 를 **하나씩** 추출한다.  
"""

In [27]:
"""
V1 - SQ code
+ 서비스 품질 요소로 그룹화
+ 서비스 품질 요소 코드
"""

prompt_v1 = """
**📄 VOC 고객경험요소 추출기  

---

### 1️⃣ 역할
당신은 **각 요약 문장**에서  

* **표준화된 상품·서비스 용어**  
* **VOC 상품·서비스 용어**  
* **표준화된 성능·품질 용어**  
* **VOC 성능·품질 용어**  
* **고객경험요소** (`표준화된 상품·서비스 용어 + 표준화된 성능·품질 용어` → 속성명사)  

를 정확히 추출하고, **제공된 서비스·품질 요소 리스트**와 가장 연관성이 높은 하나를 **매칭**합니다.  

---

### 2️⃣ 입력 형식  

| 키 | 설명 | 예시 |
|---|---|---|
| `voc요약` | 한 개 이상의 요약 문장이 개행(`\n`)으로 구분되어 들어갑니다. | `지문인식 로그인이 찾기 어렵다.` |
| `서비스·품질 요소 리스트` | 선택 가능한 요소 목록. `{ <ID>: {`name`: <요소명>, `desc`: <요소에 대한 설명>} }` 형태의 딕셔너리. | `{ "01":{"name":"로그인/인증 화면의 시인성","desc":"로그인/인증 화면은 고객이 잘 인지할 수 있도록..."}, "02":{"name":"로그인/인증 안정성", "desc":"안정적인 로그인/인증 환경을 제공하여..."}, ...}` |

---

### 3️⃣ 용어 정의  

| 구분 | 정의 | 주요 품사 | 추출 기준 |
|------|------|-----------|-----------|
| **표준화된 상품·서비스 용어** | 고객이 직접 언급한 구체적인 대상(무엇) | 명사·대명사·수사 | 가장 구체적인 명사 하나 선택 (고유명사·수량·순서 우선) |
| **VOC 상품·서비스 용어** | 원문에 그대로 나타난 대상 | 명사·대명사·수사 | 표준화된 용어와 동일 문자열 |
| **표준화된 성능·품질 용어** | 대상에 대해 고객이 평가·경험한 특성·행위(어떻게) → **속성명사** 형태 | 형용사·동사·관형사·형용사형 등 | 대상과 직접 연결된 용언 중 가장 강하게 표현된 하나 선택 (부정·긍정·정도·빈도 고려) |
| **VOC 성능·품질 용어** | 원문에 그대로 나타난 특성·행위 | 형용사·동사·관형사·형용사형 등 | 표준화된 용어와 동일 문자열 |
| **고객경험요소** | `표준화된 상품·서비스 용어 + 표준화된 성능·품질 용어` 를 **속성명사**(`‑성` 등) 형태로 만든 용어 | – | 예) `로그인` + `찾기 어렵다` → `로그인 접근성` |
| **서비스·품질 요소** | 리스트에 있는, 고객경험요소와 가장 연관성이 높은 대분류 요소 | – | 의미·키워드 매칭 후 가장 높은 점수 요소 선택 (없을 경우 빈값) |

#### 📌 속성명사 변환 규칙  

1. **형용사·동사 → ‑성**  
   * `친절하다` → `친절성`  
   * `직관적이다` → `직관성`  
   * `편리하다` → `편리성`  

2. **부정 표현 → 의미에 맞는 일반 속성명사**  
   * `찾기 어렵다` → `접근성`  
   * `느리다` → `신속성` (반대 의미를 속성명사로)  

3. **‘‑함’ 형태 금지** → 반드시 `‑성` 형태 사용  

4. **이미 ‑성 형태가 있으면 그대로 사용** (`보안성`, `안정성` 등)  

---

### 4️⃣ 추출 규칙 (절대 준수)

1. **문장별 독립 처리** – 의미 없는 문장은 무시.  
2. **표준화된 상품·서비스 용어** – 명사·대명사·수사 중 가장 구체적인 하나만 선택.  
3. **표준화된 성능·품질 용어** – 대상과 인접한 용언 중 강조·감정·빈도가 가장 높은 하나 선택.  
4. **고객경험요소 생성** – 두 표준화 용어를 속성명사 형태로 결합.  
5. **서비스·품질 요소 매칭**  
   * `cx_element`와 리스트 설명을 의미·키워드 유사도로 비교.  
   * 가장 높은 점수 요소를 `service_quality_id` 로 선택.  
   * **매칭되는 요소가 없을 경우** `service_quality_id` 를 빈값으로 반환.  
6. **결과가 전혀 없을 경우** – 전체 결과를 `"result": []` 로 반환.  

---

### 5️⃣ 출력 형식  

```json
{
    "result": [
        {
            "product_service_normalize": "<표준화된 상품·서비스 용어>",
            "product_service_voc": "<VOC 상품·서비스 용어>",
            "performence_quality_normalize": "<표준화된 성능·품질 용어>",
            "performence_quality_voc": "<VOC 성능·품질 용어>",
            "cx_element": "<속성명사 형태 고객경험요소>",
            "service_quality_id": "<서비스·품질 요소ID>"    // 매칭된 요소가 없으면 빈값
        },
        ...
    ]
}
```

* `service_quality_id` : 리스트 중 가장 연관된 요소명. 매칭되지 않으면 빈값.  

---

### 6️⃣ 예시

#### 서비스·품질 요소 리스트 (예시)
{
    `01`: {`name`: `로그인/인증 화면의 시인성`, `desc`: `로그인/인증 화면은 고객이 잘 인지할 수 있도록 명확하고 알기 쉬워야한다`},
    `02`: {`name`: `로그인/인증 안정성`, `desc`: `로그인/인증시 안정적인 로그인/인증 환경을 제공하여 시스템 오류 및 서버 문제를 최소화 해야한다`}, 
}

#### CASE 1 – 매칭 존재  
**입력**  
`voc요약`: `지문인식 로그인이 찾기 어렵다.`  

**출력**  

```json
{
    "result": [
        {
            "product_service_normalize": "지문인식",
            "product_service_voc": "지문인식 로그인",
            "performence_quality_normalize": "접근성",
            "performence_quality_voc": "찾기 어렵다",
            "cx_element": "지문인식 접근성",
            "service_quality_id": "01"
        }
    ]
}
```

#### CASE 2 – 매칭 존재  
**입력**  
`voc요약`: `인증이 가끔 렉이 걸려요.`  

**출력**  

```json
{
    "result": [
        {
            "product_service_normalize": "로그인/인증",
            "product_service_voc": "인증",
            "performence_quality_normalize": "안정성",
            "performence_quality_voc": "렉이 걸려요",
            "cx_element": "로그인/인증 안정성",
            "service_quality_id": "02"
        }
    ]
}
```

#### CASE 3 – 매칭 **없음**
**입력**  
`voc요약`: `앱 아이콘이 너무 작아요.`  

**출력**  

```json
{
    "result": [
        {
            "product_service_normalize": "앱 아이콘",
            "product_service_voc": "앱 아이콘",
            "performence_quality_normalize": "가시성",
            "performence_quality_voc": "작아요",
            "cx_element": "앱 아이콘 가시성",
            "service_quality_id": ""
        }
    ]
}
```

---

### 7️⃣ 금지 사항  

* `‑함`, `‑함` 등 속성명사가 아닌 형태 사용 금지.  
* 원문을 그대로 복사하거나 불필요한 부연을 포함하지 않음.  
* 동일 `product_service`·`performence_quality` 조합을 중복 출력하지 않음.  
* 감정 표현이 성능·품질 용어에 해당하지 않을 경우 제외.  

---

### 8️⃣ 최종 지시  

위 규칙을 **절대적으로** 따르고, **입력된 모든 요약 문장**에 대해 JSON 형식으로 결과를 반환합니다.  
결과가 하나도 없을 경우 아래와 같이 반환합니다.  

```json
{
    "result": []
}
```

---  

#### 📌 한 줄 요약 (프롬프트 사용 시 기억할 포인트)

1. **속성명사**는 반드시 `‑성`·`‑성`·`‑성` 형태 (예: 친절성, 직관성, 접근성).  
2. **문장마다** 6개의 필드를 하나씩 추출한다.  
3. `cx_element`와 **서비스·품질 요소 리스트**를 비교해 가장 연관된 요소를 `service_quality_id` 로 선택하고, 없으면 빈값으로 반환한다.  
"""

In [50]:
"""
V2 - CX요소
+ CX 요소 인입
- 상품·서비스 용어 / 성능·품질 용어
"""

prompt_v2 = """
**📄 VOC 고객경험요소 추출기  

---

### 1️⃣ 역할
당신은 **각 요약 문장**에서  

* **고객경험요소**
* **VOC 상품·서비스 용어**  
* **VOC 성능·품질 용어**  

**제공된 고객경험요소 리스트**와 가장 연관성이 높은 하나를 **매칭**하고, 용어를 정확히 추출합니다.  

---

### 2️⃣ 입력 형식  

| 키 | 설명 | 예시 |
|---|---|---|
| `고객경험요소 리스트` | 선택 가능한 고객경험요소 목록. `{ <고객경험요소>: <고객경험요소에 대한 부가설명>} }` 형태의 딕셔너리. | `{ "명확한 가이드 및 레이아웃": "처음 사용자도 쉽게 이해할 수 있는 직관적인 디자인 및 안내", ...}` |
| `voc내용` | 한 개 이상의 문장이 들어갑니다. | `지문인식 로그인이 찾기 어렵다.` |

---

### 3️⃣ 용어 정의  

| 구분 | 정의 | 주요 품사 | 추출 기준 |
|------|------|-----------|-----------|
| **고객경험요소** | 고객이 당행의 서비스를 이용하면서 경험할 수 있는 요소 | 다양한 형태 | 목록에 가장 연관되어 있는 요소 한가지를 선정한다. |
| **VOC 상품·서비스 용어** | 원문에 그대로 나타난 대상 | 명사·대명사·수사 | 표준화된 용어와 동일 문자열 |
| **VOC 성능·품질 용어** | 원문에 그대로 나타난 특성·행위 | 형용사·동사·관형사·형용사형 등 | 표준화된 용어와 동일 문자열 |

---

### 4️⃣ 추출 규칙 (절대 준수)

1. **문장별 필터링** – '결과 없음' 등 의미 없는 문장은 무시.  
2. **VOC 상품·서비스 용어** – 명사·대명사·수사 중 가장 구체적인 하나만 선택.  
3. **VOC 성능·품질 용어** – 대상과 인접한 용언 중 강조·감정·빈도가 가장 높은 하나 선택.  
3. **고객경험요소 매칭**
   * 위에서 추출된 용어를 기반으로 한다.
   * `고객경험요소 리스트`에서 요소와 설명을 이해하여 의미·키워드 유사도로 비교.  
   * 가장 높은 점수 요소를 `cx_element` 로 선택.
   * **매칭되는 요소가 없을 경우** `cx_element` 를 빈값으로 반환한다.
       -> **절때로** 없는 요소를 환각으로 만들어내지 않는다.
5. **결과가 전혀 없을 경우** – 전체 결과를 `"result": []` 로 반환.

---

### 5️⃣ 출력 형식  

```json
{
    "result": [
        {
            "voc_product_service": "<VOC 상품·서비스 용어>",
            "voc_performence_quality": "<VOC 성능·품질 용어>",
            "cx_element": "<선정된 고객경험요소>" // 매칭된 요소가 없으면 빈값
        },
        ...
    ]
}
```

* `cx_element` : **고객경험요소 리스트** 중 가장 연관된 **고객경험요소**. 매칭되지 않으면 **빈값**.  

---

### 6️⃣ 예시

**고객경험요소 리스트 (예시)**
{
    `인증 수단 구분 명확성`: `사용 가능한 인증 방식별 아이콘 및 명칭의 직관성`,
    `화면 요소의 접근성`: `고령층을 위한 충분한 크기와 색상 대비`,
    `인증 수단의 안전성 체감`: `고객이 사용하는 인증 방식의 안전성에 대한 심리적 만족도`
}

**입력**  
`voc내용`: `아이콘이 너무 작아요.\n로그인 보안이 불안해요.`  

**출력**  

```json
{
    "result": [
        {
            "voc_product_service": "아이콘",
            "voc_performence_quality": "작아요",
            "cx_element": "화면 요소의 접근성"
        },
        {
            "voc_product_service": "로그인 보안",
            "voc_performence_quality": "불안해요",
            "cx_element": "인증 수단의 안전성 체감"
        }
    ]
}
```

---

### 7️⃣ 금지 사항  

* `‑함`, `‑함` 등 속성명사가 아닌 형태 사용 금지.  
* 원문을 그대로 복사하거나 불필요한 부연을 포함하지 않음.  
* 동일 `voc_product_service`·`voc_performence_quality` 조합을 중복 출력하지 않음.  
* 감정 표현이 성능·품질 용어에 해당하지 않을 경우 제외.  
* **절대로** **고객경험요소 리스트**의 없는 요소를 환각으로 만들어내지 않음.

---

### 8️⃣ 최종 지시  

위 규칙을 **절대적으로** 따르고, **입력된 모든 VOC내용**에 대해 JSON 형식으로 결과를 반환합니다.  
결과가 하나도 없을 경우 아래와 같이 반환합니다.  

```json
{
    "result": []
}
``` 
"""

In [50]:
"""
V2 - CX요소
+ 요약이 아닌 VOC 원문
+ CX 요소 리스트 인입
"""

prompt_v2 = """
**📄 VOC 고객경험요소 추출기  

---

### 1️⃣ 역할
당신은 **각 요약 문장**에서  

* **고객경험요소**
* **VOC 상품·서비스 용어**  
* **VOC 성능·품질 용어**  

**제공된 고객경험요소 리스트**와 가장 연관성이 높은 하나를 **매칭**하고, 용어를 정확히 추출합니다.  

---

### 2️⃣ 입력 형식  

| 키 | 설명 | 예시 |
|---|---|---|
| `고객경험요소 리스트` | 선택 가능한 고객경험요소 목록. `{ <고객경험요소>: <고객경험요소에 대한 부가설명>} }` 형태의 딕셔너리. | `{ "명확한 가이드 및 레이아웃": "처음 사용자도 쉽게 이해할 수 있는 직관적인 디자인 및 안내", ...}` |
| `voc내용` | 한 개 이상의 문장이 들어갑니다. | `지문인식 로그인이 찾기 어렵다.` |

---

### 3️⃣ 용어 정의  

| 구분 | 정의 | 주요 품사 | 추출 기준 |
|------|------|-----------|-----------|
| **고객경험요소** | 고객이 당행의 서비스를 이용하면서 경험할 수 있는 요소 | 다양한 형태 | 목록에 가장 연관되어 있는 요소 한가지를 선정한다. |
| **VOC 상품·서비스 용어** | 원문에 그대로 나타난 대상 | 명사·대명사·수사 | 표준화된 용어와 동일 문자열 |
| **VOC 성능·품질 용어** | 원문에 그대로 나타난 특성·행위 | 형용사·동사·관형사·형용사형 등 | 표준화된 용어와 동일 문자열 |

---

### 4️⃣ 추출 규칙 (절대 준수)

1. **문장별 필터링** – '결과 없음' 등 의미 없는 문장은 무시.  
2. **VOC 상품·서비스 용어** – 명사·대명사·수사 중 가장 구체적인 하나만 선택.  
3. **VOC 성능·품질 용어** – 대상과 인접한 용언 중 강조·감정·빈도가 가장 높은 하나 선택.  
3. **고객경험요소 매칭**
   * 위에서 추출된 용어를 기반으로 한다.
   * `고객경험요소 리스트`에서 요소와 설명을 이해하여 의미·키워드 유사도로 비교.  
   * 가장 높은 점수 요소를 `cx_element` 로 선택.
   * **매칭되는 요소가 없을 경우** `cx_element` 를 빈값으로 반환한다.
       -> **절때로** 없는 요소를 환각으로 만들어내지 않는다.
5. **결과가 전혀 없을 경우** – 전체 결과를 `"result": []` 로 반환.

---

### 5️⃣ 출력 형식  

```json
{
    "result": [
        {
            "voc_product_service": "<VOC 상품·서비스 용어>",
            "voc_performence_quality": "<VOC 성능·품질 용어>",
            "cx_element": "<선정된 고객경험요소>" // 매칭된 요소가 없으면 빈값
        },
        ...
    ]
}
```

* `cx_element` : **고객경험요소 리스트** 중 가장 연관된 **고객경험요소**. 매칭되지 않으면 **빈값**.  

---

### 6️⃣ 예시

**고객경험요소 리스트 (예시)**
{
    `인증 수단 구분 명확성`: `사용 가능한 인증 방식별 아이콘 및 명칭의 직관성`,
    `화면 요소의 접근성`: `고령층을 위한 충분한 크기와 색상 대비`,
    `인증 수단의 안전성 체감`: `고객이 사용하는 인증 방식의 안전성에 대한 심리적 만족도`
}

**입력**  
`voc내용`: `아이콘이 너무 작아요.\n로그인 보안이 불안해요.`  

**출력**  

```json
{
    "result": [
        {
            "voc_product_service": "아이콘",
            "voc_performence_quality": "작아요",
            "cx_element": "화면 요소의 접근성"
        },
        {
            "voc_product_service": "로그인 보안",
            "voc_performence_quality": "불안해요",
            "cx_element": "인증 수단의 안전성 체감"
        }
    ]
}
```

---

### 7️⃣ 금지 사항  

* `‑함`, `‑함` 등 속성명사가 아닌 형태 사용 금지.  
* 원문을 그대로 복사하거나 불필요한 부연을 포함하지 않음.  
* 동일 `voc_product_service`·`voc_performence_quality` 조합을 중복 출력하지 않음.  
* 감정 표현이 성능·품질 용어에 해당하지 않을 경우 제외.  
* **절대로** **고객경험요소 리스트**의 없는 요소를 환각으로 만들어내지 않음.

---

### 8️⃣ 최종 지시  

위 규칙을 **절대적으로** 따르고, **입력된 모든 VOC내용**에 대해 JSON 형식으로 결과를 반환합니다.  
결과가 하나도 없을 경우 아래와 같이 반환합니다.  

```json
{
    "result": []
}
``` 
"""

In [1]:
"""
V3 - CX요소 / 매핑 Only
+ 요약이 아닌 VOC 원문
+ CX 요소 리스트 인입
- 상품·서비스 용어 / 성능·품질 용어
"""

prompt_v3 = """
**📄 VOC 고객경험요소 추출기  

---

### 1️⃣ 역할
당신은 고객 VOC(Voice of Customer) 텍스트를 분석하여, 하나의 **고객경험요소**(예: 시인성, 안정성, 편리성, 신뢰성, …)를 정확히 매칭하는 전문가 에이전트입니다.
매칭할 수 있는 **고객경험요소**는 **고객경험요소 목록**으로 제공됩니다.

---

### 2️⃣ 입력 형식  

| 키 | 설명 | 예시 |
|---|---|---|
| `고객경험요소 목록` | 선택 가능한 고객경험요소 목록. `| 고객경험요소 | 설명 |` 형태의 테이블 데이터. | `| 명확한 가이드 및 레이아웃 | 처음 사용자도 쉽게 이해할 수 있는 직관적인 디자인 및 안내 |\n| ... | ... |\n...` |
| `voc 원문` | 한 개 이상의 문장이 들어갑니다. | `지문인식 로그인이 찾기 어렵다.` |
| `설문조사채널` | VOC가 생성된 설문조사의 채널. | `플랫폼` |
| `고객경험단계구분` | 설문 문항이 응답듣고자 의도한 고객경험단계 | `로그인/인증` |

---

### 3️⃣ 용어 정의  

| 구분 | 정의 | 추출 기준 |
|------|------|-----------|
| **고객경험요소** | 고객이 당행의 서비스를 이용하면서 경험할 수 있는 요소 | 목록에 가장 연관되어 있는 요소 한가지를 선정한다. |

---

### 4️⃣ 추출 규칙 (절대 준수)

1. **전처리 – 필터링**
   * '없음', '그냥' 등 의미 없는 문장은 무시.
   * `설문조사채널`과 `고객경험단계구분`에 관계가 없다고 판단되는 VOC는 고려하지 않는다.
2. **고객경험요소 매칭**
   * `고객경험요소 목록`에서 요소와 설명을 이해하여 의미·키워드 유사도로 비교.  
   * 우선순위  
       - **부정 > 긍정** : 같은 문장에 부정·긍정 의견이 모두 포함돼 있으면 부정 의견에 해당하는 요소를 선택한다.  
       - **선언 순서** : 부정·긍정 구분이 불가능하거나 동등하게 중요할 경우, 문장에서 **먼저 언급된** 요소를 선택한다.  
   * 명확성 – 문장에서 직접적으로 언급된 키워드(예: “작다”, “느리다”, “안정적이다” 등)와 가장 연관성이 높은 요소를 매핑한다.  
       -> **절때로** 없는 요소를 환각으로 만들어내지 않는다.
3. **중복 방지** – 이미 선택된 요소가 다른 문장에 다시 등장하더라도 각 문장마다 독립적으로 판단한다.
4. **결과가 전혀 없을 경우** – 전체 결과를 `"result": ""` 로 반환.

---

### 5️⃣ 프로세스  

1. **입력 파싱**  
   - **VOC 원문**를 문장 단위(또는 의미 단위)로 분리한다.
       - `설문조사채널`과 `고객경험단계구분`에 관계가 없는 문장 또는 의미는 제거한다.
   - 각 단위에서 **키워드·감성(긍정/부정)** 을 추출한다.  
2. **고객경험요소 매칭**  
   - 추출된 키워드와 감성을 사전 정의된 **고객경험요소**와 비교하여 **연관 점수**를 산출한다.  
   - 점수가 동일하거나 판단이 모호할 경우, **선언 순서**(문장 내 위치)와 **감성 우선순위**(부정 > 긍정)를 적용한다.  
3. **최종 선택**  
   - 위 규칙에 따라 가장 높은 우선순위를 가진 **단일 요소**를 선택한다.  
   - 선택된 요소를 **정확히 하나**만 출력한다(다른 요소는 제외).  
4. **출력 형식**  
   - `result: <고개경험요소>` 형태로 반환한다.  
   - 추가 설명이나 이유는 포함하지 않는다.  
5. **예외 처리**
    - 매칭된 **고객경험요소**가 없다면, 빈값으로 반환 (`"result": ""`)

---

### 6️⃣ 출력 형식  

```json
{
    "result": "<선정된 고객경험요소>" // 매칭된 요소가 없으면 빈값
}
```

* `result` : **고객경험요소 리스트** 중 가장 연관된 **고객경험요소**. 매칭되지 않으면 **빈값**.  

---

### 7️⃣ 예시

**고객경험요소 리스트 (예시)**
| 고객경험요소 | 설명 |
|--------------|------|
| 화면 요소의 접근성      | 고령층을 위한 충분한 크기와 색상 대비 |
| 인증 수단의 안전성 체감 | 고객이 사용하는 인증 방식의 안전성에 대한 심리적 만족도 |

**입력**  
`voc내용`: `아이콘이 크고 잘보여서 좋았지만, 조금 불안해요.`  

**처리**
아이콘이 크고 잘보임 -> "화면 요소의 접근성"
불안함 -> "인증 수단의 안전성 체감"

중요도 판단 : "화면 요소의 접근성" < "인증 수단의 안전성 체감" (긍정<부정)
최종 매칭: "인증 수단의 안전성 체감"

**출력**  

```json
{
    "result": "인증 수단의 안전성 체감"
}
```

---

### 8️⃣ 금지 사항  

- **다중 요소 출력** : 하나의 VOC에 대해 두 개 이상을 제시하지 않는다.  
- **새로운 요소 생성** : 제공된 리스트에 없는 요소를 임의로 만들거나 제안하지 않는다.  
- **추가 설명** : 선택 이유, 감성 분석 결과, 점수 등 부가 정보를 출력하지 않는다. 

---

### 9️⃣ 최종 지시  

위 규칙을 **절대적으로** 따르고, **입력된 모든 VOC내용**에 대해 JSON 형식으로 결과를 반환합니다.  
결과가 하나도 없을 경우 아래와 같이 반환합니다.  

```json
{
    "result": ""
}
``` 
"""

In [13]:
"""
V4 - 복수 요소 추출 가능
+ 요약이 아닌 VOC 원문
+ CX 요소 리스트 인입
- 상품·서비스 용어 / 성능·품질 용어
+ 복수 요소 추출 가능
"""

prompt_v4 = """
**📄 VOC 고객경험요소 추출기  

---

## 1️⃣ 역할  
당신은 고객 VOC(Voice of Customer) 텍스트를 분석해 **여러 개**의 고객경험요소를 매칭하는 전문가 에이전트입니다.  
매칭 가능한 **고객경험요소**는 **고객경험요소 목록**에만 존재하며, 목록 외의 요소를 만들거나 변형해서는 안 됩니다.

---

### 2️⃣ 입력 형식  

| 키 | 설명 | 예시 |
|---|---|---|
| `고객경험요소 목록` | 선택 가능한 고객경험요소 목록. `| 고객경험요소 | 설명 |` 형태의 테이블 데이터. | `| 명확한 가이드 및 레이아웃 | 처음 사용자도 쉽게 이해할 수 있는 직관적인 디자인 및 안내 |\n| ... | ... |\n...` |
| `voc 원문` | 하나 이상의 문장(또는 의미 단위)으로 구성된 고객 의견 | `지문인식 로그인이 찾기 어렵다. 화면이 너무 작아 눈이 피로해요.` |

---

### 3️⃣ 용어 정의  

| 구분 | 정의 |
|------|------|
| **고객경험요소** | 고객이 서비스 이용 중 체감하는 요소(예: 시인성, 안정성, 편리성 등) |
| **의미 단위** | 문장 구분이 애매할 경우, “쉼표·‘그리고’·‘하지만’·‘그러나’” 등으로 구분한 **독립적인 의견 조각** |

---

### 4️⃣ 추출 규칙 (절대 준수)

1. **전처리 – 필터링**  
   * “없음”, “그냥”, “뭐든지” 등 의미가 없는 구절은 무시한다.  

2. **의미 단위 분리**  
   * `voc 원문`을 **문장** 혹은 **의미 단위**(쉼표·접속사·전환어 등) 기준으로 나눈다.  
   * 각 단위는 **독립적인 의견**이라고 가정하고 별도로 처리한다.  

3. **고객경험요소 매칭**  
   * `고객경험요소 목록`의 **요소와 설명**을 이해하고, 단위 내 **키워드·감성(긍정·부정)** 과 **유사도**를 비교한다.  
   * **우선순위**  
     1️⃣ **부정 > 긍정** – 같은 단위에 부정·긍정이 모두 포함되면 부정 의견에 해당하는 요소를 선택한다.  
     2️⃣ **선언 순서** – 부정·긍정 구분이 불가능하거나 동등할 경우, 해당 단위 내 **먼저 언급된** 요소를 선택한다.  
   * **직접 언급 키워드**(예: “작다”, “느리다”, “불안하다”)와 가장 연관된 요소를 매핑한다.  
   * **환각 금지** – 목록에 없는 요소를 만들어내지 않는다.  
   * **텍스트 변형 금지** – 매칭된 요소의 표기를 그대로 사용한다.  

4. **중복 제거**  
   * 동일 요소가 여러 단위에 매칭되더라도, **중복은 제거**한다.  

5. **결과가 전혀 없을 경우**  
   * 전체 결과를 `"result": []` 로 반환한다.  

---

### 5️⃣ 프로세스  

1. **입력 파싱**  
   - `voc 원문` → 의미 단위(문장·구) 리스트.  

2. **단위별 키워드·감성 추출**  
   - 각 단위에서 핵심 키워드와 감성(긍정/부정)을 식별한다.  

3. **요소 매칭**  
   - 키워드·감성을 기반으로 `고객경험요소 목록`과 **연관 점수**를 산출한다.  
   - 점수가 동일하거나 판단이 모호하면 **부정 > 긍정** → **선언 순서** 순으로 우선순위를 적용한다.  

4. **중요도 정렬**  
   - 매칭된 요소들을 **전체 중요도** 기준으로 정렬한다.  
   - **부정 의견**이 포함된 요소는 **긍정 의견**보다 앞에 위치한다.  
   - 동일 감성·동일 점수인 경우, **문서 내 등장 순서**(앞에서부터)대로 정렬한다.  

5. **최종 결과 구성**  
   - 정렬된 요소들을 배열 형태로 반환한다.
   - 같은 고객경험요소는 중복 제거한다.
   
6. **예외 처리**
   - 매칭된 **고객경험요소**가 없다면, 빈배열로 반환 (`"result": []`)

---

### 6️⃣ 출력 형식  

```json
{
    "result": [ "<고객경험요소1>", "<고객경험요소2>", ... ]
}
```

* `result` : **고객경험요소 목록** 중 매칭된 요소들을 **중요도 순**으로 나열한 배열.  
* 매칭된 요소가 전혀 없을 경우 → `"result": []`  

---

### 7️⃣ 예시

### 고객경험요소 리스트 (예시)

| 고객경험요소 | 설명 |
|--------------|------|
| 화면 요소의 접근성 | 고령층을 위한 충분한 크기와 색상 대비 |
| 인증 수단의 안전성 체감 | 인증 방식에 대한 심리적 안전감 |
| 응답 속도 | 서비스 응답이 빠른지 여부 |
| UI 직관성 | 사용자가 쉽게 이해·조작할 수 있는지 |

### 입력  

```json
{
    "voc 원문": "아이콘이 크고 잘 보여서 좋았지만, 로그인 과정이 불안해요. 그리고 페이지 로딩이 너무 느려서 답답합니다."
}
```

### 처리 흐름  

| 의미 단위 | 키워드·감성 | 매칭된 요소 | 비고 |
|-----------|------------|------------|------|
| 아이콘이 크고 잘 보여서 좋았지만 | “크다, 잘보이다”(긍정) | **화면 요소의 접근성** | 긍정 |
| 로그인 과정이 불안해요 | “불안”(부정) | **인증 수단의 안전성 체감** | 부정 |
| 페이지 로딩이 너무 느려서 답답합니다 | “느리다”(부정) | **응답 속도** | 부정 |

### 중요도 정렬  

1️⃣ 부정 요소 → **인증 수단의 안전성 체감**, **응답 속도**  
2️⃣ 긍정 요소 → **화면 요소의 접근성**  

### 출력  

```json
{
    "result": [
        "인증 수단의 안전성 체감",
        "응답 속도",
        "화면 요소의 접근성"
    ]
}
```

---

### 8️⃣ 금지 사항  

| 금지 내용 | 이유 |
|-----------|------|
| **새로운 요소 생성** | 제공된 리스트 외의 요소는 허용되지 않음 |
| **요소 변형** | 정확한 표기 유지 (띄어쓰기·표기법 변경 금지) |
| **다중 출력 형식** (예: 설명, 점수 등) | 출력은 오직 `result` 배열만 |
| **환각** (존재하지 않는 요소를 만들어내기) | 신뢰성 유지 |
| **중복 제거** (결과 배열에 같은 요소가 두 번 나오면) | 중복은 허용되지 않음 |

---

## 9️⃣ 최종 지시  

위 규칙을 **절대적으로** 따르고, **입력된 모든 의미 단위**에 대해 **중요도 순**으로 정렬된 고객경험요소 배열을 JSON 형태로 반환하십시오.  
매칭된 요소가 하나도 없을 경우 아래와 같이 반환합니다.  

```json
{
    "result": []
}
```
"""

In [8]:
sq_dict = {
    '로그인/인증': {
        '로그인/인증 화면의 시인성': '로그인/인증 화면은 고객이 잘 인지할 수 있도록 명확하고 알기 쉬워야한다',
        '로그인/인증의 신속성': '로그인/인증은 신속하고 원활하게 작동해야한다',
        '로그인/인증의 보안성': '로그인/인증시 보안프로그램 설치 및 다양한 보안 기능을 제공해야 한다',
        '로그인/인증 방식의 다양성': '로그인/인증의 방식은 고객의 니즈에 맞게 다양한 방식으로 선택 가능해야한다',
        '로그인/인증 안정성': '로그인/인증시 안정적인 로그인/인증 환경을 제공하여 시스템 오류 및 서버 문제를 최소화 해야한다',
        '로그인/인증 이용편의성': '로그인/인증은 생체인식 등 간편 방식을 제공하여 고객이 사용하기 편리해야 한다',
    }
}

sq_dict_with_code = {
    '로그인/인증': {
        '37': { 'name':'로그인/인증 화면의 시인성', 'desc': '로그인/인증 화면은 고객이 잘 인지할 수 있도록 명확하고 알기 쉬워야한다'},
        '38': { 'name':'로그인/인증의 신속성', 'desc': '로그인/인증은 신속하고 원활하게 작동해야한다'},
        '39': { 'name':'로그인/인증의 보안성', 'desc': '로그인/인증시 보안프로그램 설치 및 다양한 보안 기능을 제공해야 한다'},
        '40': { 'name':'로그인/인증 방식의 다양성', 'desc': '로그인/인증의 방식은 고객의 니즈에 맞게 다양한 방식으로 선택 가능해야한다'},
        '41': { 'name':'로그인/인증 안정성',  'desc': '로그인/인증시 안정적인 로그인/인증 환경을 제공하여 시스템 오류 및 서버 문제를 최소화 해야한다'},
        '42': { 'name':'로그인/인증 이용편의성', 'desc': '로그인/인증은 생체인식 등 간편 방식을 제공하여 고객이 사용하기 편리해야 한다'}
    }
}

In [9]:
cx_dict = {
    '로그인/인증': """
    | 고객경험요소 | 고객경험요소 설명 |
    |--------------|------------------|
    | '명확한 가이드 및 레이아웃' | '처음 사용자도 쉽게 이해할 수 있는 직관적인 디자인 및 안내' |
    | '인증 수단 구분 명확성' | '사용 가능한 인증 방식별 아이콘 및 명칭의 직관성' |
    | '에러 메시지의 이해도' | '오류 발생 시 원인 및 해결 방법 제시의 구체성' |
    | '화면 요소의 접근성' | '고령층을 위한 충분한 크기와 색상 대비' |
    | '입력 필드의 가독성' | '비밀번호 입력 시 피드백(마스킹 해제 옵션 등)의 적절성' |
    | '인증 소요 시간 (TTI)' | '인증 방식 선택부터 메인 화면 진입까지의 총 소요 시간 (Time To Interact)' |
    | '서버 응답 속도' | '인증 요청에 대한 서버 처리 및 응답 지연 시간' |
    | '불필요한 단계 제거' | '인증 과정 중 고객의 개입이 필요 없는 절차의 자동화' |
    | '생체 인증의 인식 속도' | '지문, 얼굴 인식 등의 생체 인증 센서 반응 속도 및 정확성' |
    | '빠른 오류 복구 시간' | '오류 발생 후 재시도 또는 다른 방식으로 전환 시 대기 시간 최소화' |
    | '인증 수단의 안전성 체감' | '고객이 사용하는 인증 방식의 안전성에 대한 심리적 만족도' |
    | '이상 거래 탐지(FDS) 알림' | '평소와 다른 환경 접속 시 즉각적인 경고 및 차단 기능의 유효성' |
    | '비밀번호/패턴 관리 용이성' | '비밀번호 변경 주기 안내 및 복잡성 요구 기준의 합리성' |
    | '앱 실행 시 보안 점검' | '앱 실행 시 보안 프로그램 작동 상태 및 점검 시간의 적절성' |
    | '다중 인증(MFA) 옵션' | '고액 거래나 민감 정보 접근 시 추가 인증 옵션의 강제성/선택성' |
    | '다양한 간편 인증 지원 범위' | '지문, 얼굴(Face ID), 간편 비밀번호, 패턴 등 시장 표준 방식 지원' |
    | '기존 인증 수단의 유지' | '공동/금융 인증서 등 기존 사용자의 익숙한 인증 방식에 대한 지원 지속' |
    | '기기 변경/재설치 시 유연성' | '기기 교체나 앱 재설치 시 인증 정보를 쉽게 가져올 수 있는 기능' |
    | '인증 방식 간의 전환 용이성' | '로그인 화면에서 여러 인증 방식을 쉽게 전환 및 선택할 수 있는 UI/UX' |
    | '비대면 신원 확인의 편리성' | '신규 가입/재등록 시 신분증 촬영, 계좌 인증 등의 간편성' |
    | '접속 오류 및 튕김 현상' | '트래픽 집중 시 발생하는 앱 강제 종료 또는 서버 접속 실패 빈도' |
    | 'OS/기기별 호환성' | '최신 및 구형 스마트폰 OS, 특정 기기 모델에서의 정상 작동 여부' |
    | '네트워크 환경 의존성' | 'Wi-Fi 및 모바일 데이터(LTE/5G) 환경 모두에서 일관된 성능 유지' |
    | '인증 실패의 비반복성' | '인증 정보를 올바르게 입력했음에도 지속적으로 실패하는 오류 방지' |
    | '시스템 점검 최소화' | '예정된 시스템 점검으로 인해 로그인 불가능한 시간 최소화' |
    | '로그인 유지 기능' | '앱을 완전히 종료하지 않았을 때 일정 시간 동안 자동 로그인 유지' |
    | '자동 로그인 설정의 용이성' | '자동 로그인 기능 설정/해제 및 관리 절차의 단순성' |
    | '인증 정보 입력의 간편성' | '간편 인증(생체/핀번호) 입력 시 최소한의 터치로 가능하게 설계' |
    | '비밀번호 찾기/재설정 흐름' | '분실 시 비대면으로 쉽고 빠르게 재설정할 수 있는 사용자 흐름' |
    | '타 서비스 연동 (SSO)' | '다른 KB 금융 앱(KB Pay 등)과의 연동을 통한 인증 절차 간소화' |
    """
}

In [3]:



voc1 = {
    'content': "휴대폰 지문등록을  변경하고 스타뱅킹 이용시  로그인 방식을  지문인식으로  변경할때  쉽게 찾을수 없었음. 인증서 관리에서 찾아들어  가야하는데 검색창에는 로그인 방법이나 지문등의 단어로는 검색할수 없었음",
    'cxCh': '플랫폼',
    'brief': "지문인식 로그인이 찾기 어렵다.",
    'cxChCd': '02',
    'cxStage': '로그인/인증',
}

voc2 = {
    'content': "중장년층이 편리하게 사용할수 있으면서 보안이 잘 되면 좋을것같다",
    'cxCh': '플랫폼',
    'brief': "인증 보안이 취약하다.",
    'cxChCd': '02',
    'cxStage': '로그인/인증',
}

voc3 = {
    'content': "로그인 화면은\n직관적이고 좋음\n다만 앱이 전반적으로 산만함",
    'cxCh': '플랫폼',
    'brief': "로그인 화면이 직관적이다.",
    'cxChCd': '02',
    'cxStage': '로그인/인증',
}

voc4 = {
    'content': "보안때문에 인증이 강화좋은데 복잡한 점도 있습니다",
    'cxCh': '플랫폼',
    'brief': "인증이 복잡하다.",
    'cxChCd': '02',
    'cxStage': '로그인/인증',
}

voc5 = {
    'content': "신체인식으로 바로 로그인 이 가능하다",
    'cxCh': '플랫폼',
    'brief': "신체인식이 가능하다.",
    'cxChCd': '02',
    'cxStage': '로그인/인증',
}

voc6 = {
    'content': "고객센터 직원이 사가지가 없다.",
    'cxCh': '플랫폼',
    'brief': "결과 없음",
    'cxChCd': '02',
    'cxStage': '로그인/인증',
}

vocs = [ 
    voc1, 
    voc2, 
    voc3, 
    voc4, 
    voc5,
    voc6
]

In [25]:
"""
V1
+ 서비스 품질 요소로 그룹화
"""

from langchain.schema import BaseMessage, HumanMessage, SystemMessage

index = 0

for voc in vocs:

    print(f"Summary Generate Index: {index} VOC:")
    print(f"{voc['brief']}")
    index = index + 1
    
    human_prompt = f"""
# 서비스·품질 요소 리스트: {sq_dict[voc['cxStage']]}
# VOC요약:
{voc['brief']}
    """.strip()

    messages = [
        SystemMessage(content=prompt_v1),
        HumanMessage(content=human_prompt)
    ]

    for chunk in llm.stream(messages):
        print(chunk.content, end="", flush=True)
    print("\n==================================================")

Summary Generate Index: 0 VOC:
지문인식 로그인이 찾기 어렵다.
```json
{
    "result": [
        {
            "product_service_normalize": "지문인식",
            "product_service_voc": "지문인식 로그인",
            "performence_quality_normalize": "접근성",
            "performence_quality_voc": "찾기 어렵다",
            "cx_element": "지문인식 접근성",
            "service_quality_id": "42"
        }
    ]
}
```
Summary Generate Index: 1 VOC:
인증 보안이 취약하다.
```json
{
    "result": [
        {
            "product_service_normalize": "인증",
            "product_service_voc": "인증 보안",
            "performence_quality_normalize": "보안성",
            "performence_quality_voc": "취약하다",
            "cx_element": "인증 보안성",
            "service_quality_id": "39"
        }
    ]
}
```
Summary Generate Index: 2 VOC:
로그인 화면이 직관적이다.
```json
{
    "result": [
        {
            "product_service_normalize": "로그인 화면",
            "product_service_voc": "로그인 화면",
            "performence_quality_normalize": "직관성",
            "performen

In [52]:
"""
V2
+ 고객경험요소 분류
"""

from langchain.schema import BaseMessage, HumanMessage, SystemMessage

index = 0

for voc in vocs:

    print(f"Summary Generate Index: {index} VOC:")
    print(f"{voc['content']}")
    index = index + 1
    
    human_prompt = f"""
# VOC내용:
{voc['content']}

---

# 고객경험요소 리스트: 
{cx_dict[voc['cxStage']]}
    """.strip()

    messages = [
        SystemMessage(content=prompt_v2),
        HumanMessage(content=human_prompt)
    ]

    for chunk in llm.stream(messages):
        print(chunk.content, end="", flush=True)
    print("\n==================================================")

Summary Generate Index: 0 VOC:
휴대폰 지문등록을  변경하고 스타뱅킹 이용시  로그인 방식을  지문인식으로  변경할때  쉽게 찾을수 없었음. 인증서 관리에서 찾아들어  가야하는데 검색창에는 로그인 방법이나 지문등의 단어로는 검색할수 없었음
```json
{
    "result": [
        {
            "voc_product_service": "지문인식",
            "voc_performence_quality": "찾을 수 없었음",
            "cx_element": "인증 방식 간의 전환 용이성"
        },
        {
            "voc_product_service": "검색창",
            "voc_performence_quality": "검색할 수 없었음",
            "cx_element": "인증 수단 구분 명확성"
        }
    ]
}
```
Summary Generate Index: 1 VOC:
중장년층이 편리하게 사용할수 있으면서 보안이 잘 되면 좋을것같다
```json
{
    "result": [
        {
            "voc_product_service": "보안",
            "voc_performence_quality": "잘 되다",
            "cx_element": "인증 수단의 안전성 체감"
        }
    ]
}
```
Summary Generate Index: 2 VOC:
로그인 화면은
직관적이고 좋음
다만 앱이 전반적으로 산만함
```json
{
    "result": [
        {
            "voc_product_service": "로그인 화면",
            "voc_performence_quality": "직관적",
            "cx_element": "명확한 가이드 및 레이아웃"
        },
 

In [12]:
"""
V3
+ 고객경험요소 분류
- 개체어 제거
"""

from langchain.schema import BaseMessage, HumanMessage, SystemMessage

index = 0

for voc in vocs:

    print(f"Summary Generate Index: {index} VOC:")
    print(f"{voc['content']}")
    index = index + 1
    
    human_prompt = f"""
# VOC 정보:
## VOC 원문:
{voc['content']}

## 설문조사채널:
{voc['cxCh']}

## 고객경험단계:
{voc['cxStage']}

---

# 고객경험요소 목록: 
{cx_dict[voc['cxStage']]}
    """.strip()

    messages = [
        SystemMessage(content=prompt_v3),
        HumanMessage(content=human_prompt)
    ]

    for chunk in llm.stream(messages):
        print(chunk.content, end="", flush=True)
    print("\n==================================================")

Summary Generate Index: 0 VOC:
휴대폰 지문등록을  변경하고 스타뱅킹 이용시  로그인 방식을  지문인식으로  변경할때  쉽게 찾을수 없었음. 인증서 관리에서 찾아들어  가야하는데 검색창에는 로그인 방법이나 지문등의 단어로는 검색할수 없었음
{
    "result": "인증 방식 간의 전환 용이성"
}
Summary Generate Index: 1 VOC:
중장년층이 편리하게 사용할수 있으면서 보안이 잘 되면 좋을것같다
{
    "result": "화면 요소의 접근성"
}
Summary Generate Index: 2 VOC:
로그인 화면은
직관적이고 좋음
다만 앱이 전반적으로 산만함
```json
{
    "result": "명확한 가이드 및 레이아웃"
}
```
Summary Generate Index: 3 VOC:
보안때문에 인증이 강화좋은데 복잡한 점도 있습니다
{
    "result": "불필요한 단계 제거"
}
Summary Generate Index: 4 VOC:
신체인식으로 바로 로그인 이 가능하다
{
    "result": "생체 인증의 인식 속도"
}
Summary Generate Index: 5 VOC:
고객센터 직원이 사가지가 없다.
```json
{
    "result": ""
}
```


In [14]:
"""
V4
+ 고객경험요소 분류
+ 복수 결과
"""

from langchain.schema import BaseMessage, HumanMessage, SystemMessage

index = 0

for voc in vocs:

    print(f"Summary Generate Index: {index} VOC:")
    print(f"{voc['content']}")
    index = index + 1
    
    human_prompt = f"""
# VOC 정보:
## VOC 원문:
{voc['content']}

---

# 고객경험요소 목록: 
{cx_dict[voc['cxStage']]}
    """.strip()

    messages = [
        SystemMessage(content=prompt_v4),
        HumanMessage(content=human_prompt)
    ]

    for chunk in llm.stream(messages):
        print(chunk.content, end="", flush=True)
    print("\n==================================================")

Summary Generate Index: 0 VOC:
휴대폰 지문등록을  변경하고 스타뱅킹 이용시  로그인 방식을  지문인식으로  변경할때  쉽게 찾을수 없었음. 인증서 관리에서 찾아들어  가야하는데 검색창에는 로그인 방법이나 지문등의 단어로는 검색할수 없었음
```json
{
    "result": [
        "명확한 가이드 및 레이아웃",
        "불필요한 단계 제거"
    ]
}
```
Summary Generate Index: 1 VOC:
중장년층이 편리하게 사용할수 있으면서 보안이 잘 되면 좋을것같다
```json
{
    "result": [
        "불필요한 단계 제거",
        "인증 수단의 안전성 체감"
    ]
}
```
Summary Generate Index: 2 VOC:
로그인 화면은
직관적이고 좋음
다만 앱이 전반적으로 산만함
```json
{
    "result": [
        "명확한 가이드 및 레이아웃"
    ]
}
```
Summary Generate Index: 3 VOC:
보안때문에 인증이 강화좋은데 복잡한 점도 있습니다
```json
{
    "result": [
        "불필요한 단계 제거",
        "인증 수단의 안전성 체감"
    ]
}
```
Summary Generate Index: 4 VOC:
신체인식으로 바로 로그인 이 가능하다
```json
{
    "result": [
        "생체 인증의 인식 속도"
    ]
}
```
Summary Generate Index: 5 VOC:
고객센터 직원이 사가지가 없다.
```json
{
    "result": []
}
```
